In [55]:

#単純にDBに接続してカラム数をカウントしてみる
import pandas as pd
import sqlite3

conn= sqlite3.connect("all_stocks.db")
query = """
PRAGMA table_info(stock_price)
"""
key_df =pd.read_sql(query,conn)
display(key_df)

,cid,name,type,notnull,dflt_value,pk
0,0,ticker,TEXT,0,None,1
1,1,date,TEXT,0,None,2
2,2,open,REAL,0,None,0
3,3,high,REAL,0,None,0
4,4,low,REAL,0,None,0
5,5,close,REAL,0,None,0
6,6,volume,INTEGER,0,None,0


In [59]:
import sqlite3
import pandas as pd

conn=sqlite3.connect("all_stocks.db")

query="""
SELECT COUNT(*) as cnt
FROM stock_price"""

pd.read_sql(query,conn)

,cnt
0,6934822


In [50]:
last_df.isna().sum()
last_df["date"].max()
last_df["ticker"].nunique()

3749

In [ ]:
import pandas as pd
import yfinance as yf

df=yf.download("7203.T",start="2025-01-01")

df.columns=df.columns.get_level_values(0)
df.columns.name = None
df.reset_index(inplace=True)
display(df.head(2))

[*********************100%***********************]  1 of 1 completed


,Date,Close,High,Low,Open,Volume
0,2025-01-06,2870.214600,2958.866196,2859.728927,2957.912953,41298500
1,2025-01-07,2909.297363,2980.790582,2854.485896,2863.541704,44700900


In [34]:
import pandas as pd
import sqlite3



conn= sqlite3.connect("all_stocks.db")
tickers="7203.T"
query = f"""SELECT MAX(date) as max_date FROM stock_price WHERE ticker='{tickers}'"""

last_date_df=pd.read_sql(query,conn)

last_date = last_date_df.loc[0, "max_date"]

if pd.isna(last_date):
    start_date = "2018-01-01"
else:
    start_date = (
    pd.to_datetime(last_date)
    + pd.Timedelta(days=1)
    ).strftime("%Y-%m-%d")


df = yf.download(
        tickers,
        start=start_date,
        progress=False
        )

if df.empty:
    print("追加データなし")

else:
    print("新規データあり")
    display(df.head())

新規データあり


Price,Close,High,Low,Open,Volume
Ticker,7203.T,7203.T,7203.T,7203.T,7203.T
Date,,,,,
2026-04-13,3319.0,3337.0,3280.0,3295.0,14353200
2026-04-14,3326.0,3355.0,3288.0,3355.0,15920200
2026-04-15,3381.0,3386.0,3344.0,3355.0,20317000


In [80]:
import sqlite3
import pandas as pd
import yfinance as yf
DB_PATH = "all_stocks.db"

def update_stock(ticker):

    # DB接続
    conn = sqlite3.connect(DB_PATH)

    # 最新日付取得
    query = f"""
    SELECT MAX(date) as max_date
    FROM stock_price
    WHERE ticker = '{ticker}'
    """
    last_date_df = pd.read_sql(query, conn)
    last_date = last_date_df.loc[0, "max_date"]

    # 開始日決定
    if pd.isna(last_date):
        start_date = "2018-01-01"
    else:
        start_date = (
            pd.to_datetime(last_date) + pd.Timedelta(days=1)
        ).strftime("%Y-%m-%d")

    print("取得開始日:", start_date)

    # データ取得
    df = yf.download(ticker, start=start_date, progress=False)

    if df.empty:
        print("追加データなし")
        conn.close()

        return

    # カラム整理
    df.columns = df.columns.get_level_values(0)
    df.columns.name = None

    df["ticker"] = ticker
    df = df.reset_index()

    # 🔥 日付フォーマット統一（超重要）
    df["Date"] = pd.to_datetime(df["Date"]).dt.strftime("%Y-%m-%d")

    # カラム名統一
    df = df.rename(columns={
        "Date": "date",
        "Open": "open",
        "High": "high",
        "Low": "low",
        "Close": "close",
        "Volume": "volume"
    })

    df = df[["ticker", "date", "open", "high", "low", "close", "volume"]]


     # =====================
        # ⑤ 重複除去（最重要）
     # =====================
    existing_dates = pd.read_sql(f"""
    SELECT date FROM stock_price WHERE ticker = '{ticker}'
    """, conn)

    df = df[~df["date"].isin(existing_dates["date"])]

    if df.empty:
        print(f"{ticker}: 追加なし（重複）")
        conn.close()
        return

    print(f"{ticker}: {len(df)}件追加")


    # DBに追加
    df.to_sql(
        "stock_price",
        conn,
        if_exists="append",
        index=False
    )

    print("DB更新完了:", ticker)

    conn.commit()
    conn.close()


# =====================
# メイン処理
# =====================
def main():

    conn = sqlite3.connect(DB_PATH)

    tickers_df = pd.read_sql("""
    SELECT コード
    FROM stock_master
    WHERE 市場・商品区分 LIKE '%プライム%'
   OR 市場・商品区分 LIKE '%スタンダード%'
   OR 市場・商品区分 LIKE '%グロース%'
    """, conn)

    conn.close()

    tickers = [f"{code}.T" for code in tickers_df["コード"].tolist()]

    print(f"対象銘柄数: {len(tickers)}")

    for ticker in tickers:
        update_stock(ticker)


if __name__ == "__main__":
    main()

対象銘柄数: 3773
取得開始日: 2026-04-18
1301.T: 追加なし（重複）
取得開始日: 2026-04-18
130A.T: 追加なし（重複）
取得開始日: 2026-04-18
1332.T: 追加なし（重複）
取得開始日: 2026-04-18
1333.T: 追加なし（重複）
取得開始日: 2026-04-18
135A.T: 追加なし（重複）
取得開始日: 2026-04-18
1375.T: 追加なし（重複）
取得開始日: 2026-04-18
1376.T: 追加なし（重複）
取得開始日: 2026-04-18
1377.T: 追加なし（重複）
取得開始日: 2026-04-18
1379.T: 追加なし（重複）
取得開始日: 2026-04-18
137A.T: 追加なし（重複）
取得開始日: 2026-04-18


$1380.T: possibly delisted; no price data found  (1d 2026-04-18 -> 2026-04-18)

1 Failed download:
['1380.T']: possibly delisted; no price data found  (1d 2026-04-18 -> 2026-04-18)


追加データなし
取得開始日: 2026-04-18
1381.T: 追加なし（重複）
取得開始日: 2026-04-18
1382.T: 追加なし（重複）
取得開始日: 2026-04-18
1383.T: 追加なし（重複）
取得開始日: 2026-04-18
1384.T: 追加なし（重複）
取得開始日: 2026-04-18
138A.T: 追加なし（重複）
取得開始日: 2026-04-18
1401.T: 追加なし（重複）
取得開始日: 2026-04-18
1407.T: 追加なし（重複）
取得開始日: 2026-04-18
1414.T: 追加なし（重複）
取得開始日: 2026-04-18
1417.T: 追加なし（重複）
取得開始日: 2026-04-18


KeyboardInterrupt: 

In [74]:
import pandas as pd
import sqlite3

conn= sqlite3.connect("all_stocks.db")
query = """
PRAGMA table_info(stock_price)
"""
key_df =pd.read_sql(query,conn)
display(key_df)


,cid,name,type,notnull,dflt_value,pk
0,0,ticker,TEXT,0,None,1
1,1,date,TEXT,0,None,2
2,2,open,REAL,0,None,0
3,3,high,REAL,0,None,0
4,4,low,REAL,0,None,0
5,5,close,REAL,0,None,0
6,6,volume,INTEGER,0,None,0
